# CS5224 Data Curation — Stage 2B

## Cross-retailer candidate generation for the pilot categories

Stage 2A selected a cross-category pilot and produced a clean catalog candidate pool.

Stage 2B generates **cross-retailer candidate pairs** for later exact verification.

### Important rule

Stage 2B does **not** automatically approve a canonical product match.

A candidate pair is proposed only when the records have:

- the same normalized brand;
- compatible exact pack structure for the strict candidate path;
- the same target category;
- sufficient product-name evidence to make manual verification worthwhile.

The final exact-match decision remains a verification task because variant wording can differ across retailers.

### Stock treatment

Product identity is independent of current stock.

Therefore:

- catalog-eligible in-stock records can be matched;
- catalog-eligible out-of-stock records can also be matched;
- `offer_eligible` remains separate and indicates whether the listing is currently purchasable.

### Stage 2B outputs

The notebook produces:

- strict candidate pairs;
- high-priority review candidates;
- pack-discrepancy candidates;
- block summaries;
- unmatched pilot listings;
- per-category/per-retailer candidate summaries.

No Stage 2A values are overwritten.


## 1. Imports and project paths

In [1]:
from pathlib import Path
from difflib import SequenceMatcher
import math
import re

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
CURATED_DATA_DIR = PROJECT_ROOT / "data" / "curated"

PILOT_POOL_FILE = (
    CURATED_DATA_DIR
    / "stage2a_pilot_candidate_pool.csv"
)

PILOT_CATEGORIES_FILE = (
    CURATED_DATA_DIR
    / "stage2a_pilot_categories.csv"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nPilot pool:")
print(PILOT_POOL_FILE)

print("\nPilot categories:")
print(PILOT_CATEGORIES_FILE)

assert PILOT_POOL_FILE.exists(), (
    "Stage 2A pilot candidate pool was not found."
)

assert PILOT_CATEGORIES_FILE.exists(), (
    "Stage 2A pilot category file was not found."
)


Project root:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix

Pilot pool:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_pilot_candidate_pool.csv

Pilot categories:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_pilot_categories.csv


## 2. Load Stage 2A pilot data

In [2]:
pilot = pd.read_csv(
    PILOT_POOL_FILE,
    low_memory=False,
)

pilot_categories_df = pd.read_csv(
    PILOT_CATEGORIES_FILE,
    low_memory=False,
)

print("Pilot candidate pool shape:")
print(pilot.shape)

print("\nSelected pilot categories:")
display(
    pilot_categories_df[
        [
            column
            for column in [
                "pilot_rank",
                "target_category",
                "catalog_total",
                "offer_total",
            ]
            if column in pilot_categories_df.columns
        ]
    ]
)

display(pilot.head(3))


Pilot candidate pool shape:
(31595, 55)

Selected pilot categories:


,pilot_rank,target_category,catalog_total,offer_total
0,1,Snacks,15196,11551
1,2,Drinks,9404,7383
2,3,Canned & Packaged Food,6995,6073


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,source_category_normalized,product_name_raw,product_name_normalized,brand_raw,...,eligibility_status,source_category_match_count,product_name_match_count,target_category_suggested,target_category_final,category_mapping_status,category_mapping_confidence,category_mapping_method,source_category_candidates,product_name_candidates
0,redmart,redmart_all_products_final.xlsx,Sheet1,2,redmart_00000002,shop-chilled-desserts,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,sojasun natural soya yoghurt,Sojasun,...,ELIGIBLE_PRELIMINARY,1,1,Snacks,Snacks,AUTO_SOURCE_CATEGORY,HIGH,SOURCE_CATEGORY_RULE,Snacks,Dairy
1,redmart,redmart_all_products_final.xlsx,Sheet1,3,redmart_00000003,shop-chilled-desserts,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix chocolate custard dessert cream 4x125g,Monoprix,...,ELIGIBLE_PRELIMINARY,1,1,Snacks,Snacks,AUTO_SOURCE_CATEGORY,HIGH,SOURCE_CATEGORY_RULE,Snacks,Snacks
2,redmart,redmart_all_products_final.xlsx,Sheet1,4,redmart_00000004,shop-chilled-desserts,shop-chilled-desserts,Top Gourmet Cheng Tng,top gourmet cheng tng,Top Gourmet,...,ELIGIBLE_PRELIMINARY,1,0,Snacks,Snacks,AUTO_SOURCE_CATEGORY,HIGH,SOURCE_CATEGORY_RULE,Snacks,NONE


## 3. Restore important nullable data types

In [3]:
def to_nullable_boolean(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    mapping = {
        "true": True,
        "false": False,
        "yes": True,
        "no": False,
        "1": True,
        "0": False,
    }

    return (
        normalized
        .map(mapping)
        .astype("boolean")
    )


for column in [
    "in_stock",
    "currently_available",
    "has_offer",
    "needs_manual_review",
    "basic_eligibility",
    "catalog_eligible",
    "offer_eligible",
]:
    if column in pilot.columns:
        pilot[column] = to_nullable_boolean(
            pilot[column]
        )


if "pack_count" in pilot.columns:
    pilot["pack_count"] = (
        pd.to_numeric(
            pilot["pack_count"],
            errors="coerce"
        )
        .astype("Int64")
    )


for column in [
    "unit_size",
    "total_quantity",
    "total_quantity_base",
    "product_price",
    "original_price",
]:
    if column in pilot.columns:
        pilot[column] = (
            pd.to_numeric(
                pilot[column],
                errors="coerce"
            )
            .astype("Float64")
        )


## 4. Validate pilot input

In [4]:
REQUIRED_COLUMNS = [
    "target_category_final",
    "retailer",
    "source_record_id",
    "product_name_raw",
    "product_name_normalized",
    "brand_normalized",
    "specification_raw",
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
    "product_price",
    "stock_status",
    "currently_available",
    "catalog_eligible",
    "offer_eligible",
    "product_url_raw",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in pilot.columns
]

assert not missing_columns, (
    f"Missing required Stage 2B columns: {missing_columns}"
)

assert (
    pilot["source_record_id"]
    .isna()
    .sum()
    == 0
)

assert (
    pilot["source_record_id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    pilot["catalog_eligible"]
    .fillna(False)
    .all()
), (
    "Stage 2B input should contain only catalog-eligible records."
)

assert (
    pilot["brand_normalized"]
    .notna()
    .all()
), (
    "Pilot catalog records should have normalized brands."
)

assert (
    pilot["target_category_final"]
    .notna()
    .all()
)

selected_pilot_categories = (
    pilot_categories_df["target_category"]
    .dropna()
    .astype(str)
    .tolist()
)

assert set(
    pilot["target_category_final"]
    .dropna()
    .unique()
) <= set(selected_pilot_categories)

print("PASS: Stage 2B pilot input validation succeeded.")

print("\nPilot rows by category and retailer:")
display(
    pd.crosstab(
        pilot["target_category_final"],
        pilot["retailer"],
        margins=True,
    )
)


PASS: Stage 2B pilot input validation succeeded.

Pilot rows by category and retailer:


retailer,coldstorage,fairprice,redmart,All
target_category_final,,,,
Canned & Packaged Food,628,1357,5010,6995
Drinks,1040,2145,6219,9404
Snacks,1262,3547,10387,15196
All,2930,7049,21616,31595


## 5. Preserve Stage 2A pilot data

In [5]:
pilot_snapshot = pilot.copy(deep=True)

print(
    f"Pilot snapshot created: "
    f"{len(pilot_snapshot):,} rows"
)


Pilot snapshot created: 31,595 rows


## 6. Compute a canonical per-unit base quantity

The strict pack signature distinguishes, for example:

- `1 × 600 g`
- `12 × 50 g`

even though both total 600 g.

This is required because an equal total quantity does not necessarily mean an exact pack match.


In [6]:
pilot2b = pilot.copy()

pilot2b["per_unit_quantity_base"] = (
    pilot2b["total_quantity_base"]
    /
    pilot2b["pack_count"].astype("Float64")
)

# Round only for blocking/key stability.
# Source values are preserved in their original columns.
pilot2b["per_unit_quantity_base_key"] = (
    pilot2b["per_unit_quantity_base"]
    .round(6)
)

pilot2b["total_quantity_base_key"] = (
    pilot2b["total_quantity_base"]
    .round(6)
)

assert (
    pilot2b[
        "per_unit_quantity_base"
    ] > 0
).all()

print(
    "PASS: Per-unit base quantities created."
)


PASS: Per-unit base quantities created.


## 7. Create strict and relaxed pack keys

In [7]:
def safe_key_value(value):
    if pd.isna(value):
        return "<NA>"

    if isinstance(value, (float, np.floating)):
        return f"{float(value):.6f}"

    return str(value)


pilot2b["strict_pack_key"] = (
    pilot2b.apply(
        lambda row: "|".join(
            [
                safe_key_value(
                    row["base_unit"]
                ),
                safe_key_value(
                    row["pack_count"]
                ),
                safe_key_value(
                    row[
                        "per_unit_quantity_base_key"
                    ]
                ),
                safe_key_value(
                    row[
                        "total_quantity_base_key"
                    ]
                ),
            ]
        ),
        axis=1,
    )
    .astype("string")
)

# Relaxed key is used only to find possible pack-description
# discrepancies. It is NOT sufficient for an exact match.
pilot2b["total_pack_key"] = (
    pilot2b.apply(
        lambda row: "|".join(
            [
                safe_key_value(
                    row["base_unit"]
                ),
                safe_key_value(
                    row[
                        "total_quantity_base_key"
                    ]
                ),
            ]
        ),
        axis=1,
    )
    .astype("string")
)

print("Example strict pack keys:")
display(
    pilot2b[
        [
            "specification_raw",
            "pack_count",
            "total_quantity_base",
            "base_unit",
            "per_unit_quantity_base",
            "strict_pack_key",
        ]
    ].head(10)
)


Example strict pack keys:


,specification_raw,pack_count,total_quantity_base,base_unit,per_unit_quantity_base,strict_pack_key
0,6 × 100 g,6,600.0,g,100.0,g|6|100.000000|600.000000
1,4 × 125 g,4,500.0,g,125.0,g|4|125.000000|500.000000
2,400 g,1,400.0,g,400.0,g|1|400.000000|400.000000
3,4 × 115 g,4,460.0,g,115.0,g|4|115.000000|460.000000
4,4 × 100 g,4,400.0,g,100.0,g|4|100.000000|400.000000
5,1 kg,1,1000.0,g,1000.0,g|1|1000.000000|1000.000000
6,430 g,1,430.0,g,430.0,g|1|430.000000|430.000000
7,200 g,1,200.0,g,200.0,g|1|200.000000|200.000000
8,360 g,1,360.0,g,360.0,g|1|360.000000|360.000000
9,200 g,1,200.0,g,200.0,g|1|200.000000|200.000000


## 8. Build a product-name comparison representation

Brand and obvious pack-size notation are removed from the comparison string.

The original product names remain unchanged.

Variant terms such as `zero sugar`, `original`, `lemon`, `chocolate`, `salted`, etc. remain in the text because they are essential for exact product identity.


In [8]:
MATCH_STOPWORDS = {
    "the",
    "and",
    "with",
    "for",
    "of",
    "by",
    "a",
    "an",
    "pack",
    "packs",
    "pkt",
    "ctn",
    "carton",
    "pcs",
    "pc",
    "piece",
    "pieces",
    "per",
    "each",
    "x",
    "g",
    "kg",
    "mg",
    "ml",
    "l",
}


def normalize_match_text(
    product_name,
    brand
):
    if pd.isna(product_name):
        return ""

    text = str(product_name).casefold()

    # Remove normalized brand string when it appears literally.
    if (
        not pd.isna(brand)
        and str(brand).strip()
    ):
        brand_text = (
            str(brand)
            .casefold()
            .strip()
        )

        text = text.replace(
            brand_text,
            " "
        )

    # Remove common metric/count pack patterns only.
    text = re.sub(
        r"\b\d+(?:\.\d+)?\s*"
        r"(?:x|×)\s*"
        r"\d+(?:\.\d+)?\s*"
        r"(?:mg|g|kg|ml|l|pcs?|pieces?)\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\b\d+(?:\.\d+)?\s*"
        r"(?:mg|g|kg|ml|l|pcs?|pieces?)\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text
    )

    tokens = [
        token
        for token in text.split()
        if token not in MATCH_STOPWORDS
    ]

    return " ".join(tokens)


pilot2b["name_match_text"] = [
    normalize_match_text(
        product_name,
        brand,
    )
    for product_name, brand
    in zip(
        pilot2b[
            "product_name_normalized"
        ],
        pilot2b[
            "brand_normalized"
        ],
    )
]

pilot2b["name_match_tokens"] = (
    pilot2b["name_match_text"]
    .apply(
        lambda text: tuple(
            sorted(
                set(
                    str(text).split()
                )
            )
        )
    )
)

display(
    pilot2b[
        [
            "product_name_raw",
            "brand_normalized",
            "name_match_text",
        ]
    ].head(20)
)


,product_name_raw,brand_normalized,name_match_text
0,Sojasun Natural Soya Yoghurt,sojasun,natural soya yoghurt
1,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix,chocolate custard dessert cream
2,Top Gourmet Cheng Tng,top gourmet,cheng tng
3,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,la laitiere,vanilla rice pudding riz au lait
4,Monoprix Flan Caramel Vanilla Pudding 4X100G,monoprix,flan caramel vanilla pudding
5,Fortune Sweetened Chin Chow 1KG,fortune,sweetened chin chow
6,RedMart Trio Swiss Roll (Singapore Special),redmart,trio swiss roll singapore special
7,Arla Protein Chocolate Pudding 200G - Lactose ...,arla,protein chocolate pudding lactose free
8,Fortune Soya Beancurd Almond 360G,fortune,soya beancurd almond
9,Arla Protein Hazelnut Pudding 200G - Lactose Free,arla,protein hazelnut pudding lactose free


## 9. Define name-similarity functions

In [9]:
def token_jaccard(
    left_text,
    right_text
):
    left = set(
        str(left_text).split()
    )
    right = set(
        str(right_text).split()
    )

    if not left and not right:
        return 1.0

    if not left or not right:
        return 0.0

    return (
        len(left & right)
        /
        len(left | right)
    )


def sequence_similarity(
    left_text,
    right_text
):
    return SequenceMatcher(
        None,
        str(left_text),
        str(right_text),
    ).ratio()


def score_name_pair(
    left_text,
    right_text
):
    jaccard = token_jaccard(
        left_text,
        right_text
    )

    sequence = sequence_similarity(
        left_text,
        right_text
    )

    exact_core = (
        str(left_text)
        == str(right_text)
        and str(left_text) != ""
    )

    combined = (
        0.60 * jaccard
        + 0.40 * sequence
    )

    return (
        jaccard,
        sequence,
        combined,
        exact_core,
    )


## 10. Define retailer pairs

Candidate matching is performed separately for each retailer pair.


In [10]:
RETAILER_PAIRS = [
    ("redmart", "fairprice"),
    ("redmart", "coldstorage"),
    ("fairprice", "coldstorage"),
]

available_retailers = set(
    pilot2b["retailer"]
    .dropna()
    .unique()
)

print("Retailers in pilot:")
print(sorted(available_retailers))

for left, right in RETAILER_PAIRS:
    assert left in available_retailers
    assert right in available_retailers


Retailers in pilot:
['coldstorage', 'fairprice', 'redmart']


## 11. Build strict candidate blocks

Strict blocking requires exact equality on:

- target category;
- normalized brand;
- strict pack key.

This dramatically reduces the candidate search space before any name similarity is considered.


In [11]:
STRICT_BLOCK_COLUMNS = [
    "target_category_final",
    "brand_normalized",
    "strict_pack_key",
]

strict_block_summary = (
    pilot2b
    .groupby(
        STRICT_BLOCK_COLUMNS,
        dropna=False,
    )
    .agg(
        total_records=(
            "source_record_id",
            "size"
        ),
        retailer_count=(
            "retailer",
            "nunique"
        ),
        redmart_records=(
            "retailer",
            lambda s: int(
                (s == "redmart").sum()
            )
        ),
        fairprice_records=(
            "retailer",
            lambda s: int(
                (s == "fairprice").sum()
            )
        ),
        coldstorage_records=(
            "retailer",
            lambda s: int(
                (s == "coldstorage").sum()
            )
        ),
    )
    .reset_index()
)

multi_retailer_blocks = (
    strict_block_summary.loc[
        strict_block_summary[
            "retailer_count"
        ] >= 2
    ]
    .copy()
)

print(
    "Strict pack/brand/category blocks:",
    f"{len(strict_block_summary):,}"
)

print(
    "Multi-retailer strict blocks:",
    f"{len(multi_retailer_blocks):,}"
)

display(
    multi_retailer_blocks
    .sort_values(
        "total_records",
        ascending=False
    )
    .head(20)
)


Strict pack/brand/category blocks: 13,341
Multi-retailer strict blocks: 1,423


,target_category_final,brand_normalized,strict_pack_key,total_records,retailer_count,redmart_records,fairprice_records,coldstorage_records
12353,Snacks,snackfirst,g|1|200.000000|200.000000,123,2,91,32,0
10201,Snacks,laobanniang,g|1|500.000000|500.000000,63,2,40,23,0
6012,Drinks,pokka,ml|1|1500.000000|1500.000000,56,3,21,20,15
6024,Drinks,pokka,ml|6|250.000000|1500.000000,55,3,22,23,10
12350,Snacks,snackfirst,g|1|150.000000|150.000000,51,2,35,16,0
6016,Drinks,pokka,ml|1|500.000000|500.000000,49,3,14,3,32
11810,Snacks,redman,g|1|1000.000000|1000.000000,44,2,3,41,0
10347,Snacks,lindt,g|1|100.000000|100.000000,43,3,25,9,9
12349,Snacks,snackfirst,g|1|120.000000|120.000000,40,2,22,18,0
11912,Snacks,ritter sport,g|1|100.000000|100.000000,40,2,21,19,0


## 12. Candidate-generation safeguards

Most strict blocks are small. For unusually large blocks, exhaustive Cartesian pairing can create too many low-value comparisons.

For a large block, Stage 2B first requires the two product names to share at least one meaningful token.

This only reduces candidate generation; it never approves a match.


In [12]:
MAX_CARTESIAN_PAIRS_PER_BLOCK = 5_000
MAX_CANDIDATES_PER_SOURCE_PER_PAIR = 5
MIN_COMBINED_SIMILARITY_TO_KEEP = 0.25


def shared_meaningful_token(
    left_text,
    right_text
):
    left = set(
        str(left_text).split()
    )

    right = set(
        str(right_text).split()
    )

    return bool(
        left & right
    )


## 13. Generate strict cross-retailer candidate pairs

In [13]:
PAIR_OUTPUT_COLUMNS = [
    "candidate_pair_id",
    "retailer_pair",
    "target_category",
    "brand_normalized",
    "strict_pack_key",

    "left_retailer",
    "left_source_record_id",
    "left_product_name",
    "left_name_match_text",
    "left_specification",
    "left_pack_count",
    "left_total_quantity_base",
    "left_base_unit",
    "left_product_price",
    "left_stock_status",
    "left_offer_eligible",
    "left_product_url",

    "right_retailer",
    "right_source_record_id",
    "right_product_name",
    "right_name_match_text",
    "right_specification",
    "right_pack_count",
    "right_total_quantity_base",
    "right_base_unit",
    "right_product_price",
    "right_stock_status",
    "right_offer_eligible",
    "right_product_url",

    "token_jaccard",
    "sequence_similarity",
    "combined_similarity",
    "exact_name_core",
    "candidate_priority",
]


def make_pair_record(
    left_row,
    right_row,
    left_retailer,
    right_retailer,
):
    (
        jaccard,
        sequence,
        combined,
        exact_core,
    ) = score_name_pair(
        left_row["name_match_text"],
        right_row["name_match_text"],
    )

    if exact_core:
        priority = "VERY_HIGH"

    elif (
        jaccard >= 0.80
        and sequence >= 0.80
    ):
        priority = "HIGH"

    elif combined >= 0.65:
        priority = "MEDIUM"

    else:
        priority = "LOW"

    left_id = str(
        left_row[
            "source_record_id"
        ]
    )

    right_id = str(
        right_row[
            "source_record_id"
        ]
    )

    pair_id = (
        f"{left_id}__{right_id}"
    )

    return {
        "candidate_pair_id": pair_id,
        "retailer_pair": (
            f"{left_retailer}"
            f"__{right_retailer}"
        ),
        "target_category": left_row[
            "target_category_final"
        ],
        "brand_normalized": left_row[
            "brand_normalized"
        ],
        "strict_pack_key": left_row[
            "strict_pack_key"
        ],

        "left_retailer": left_retailer,
        "left_source_record_id": left_id,
        "left_product_name": left_row[
            "product_name_raw"
        ],
        "left_name_match_text": left_row[
            "name_match_text"
        ],
        "left_specification": left_row[
            "specification_raw"
        ],
        "left_pack_count": left_row[
            "pack_count"
        ],
        "left_total_quantity_base": left_row[
            "total_quantity_base"
        ],
        "left_base_unit": left_row[
            "base_unit"
        ],
        "left_product_price": left_row[
            "product_price"
        ],
        "left_stock_status": left_row[
            "stock_status"
        ],
        "left_offer_eligible": left_row[
            "offer_eligible"
        ],
        "left_product_url": left_row[
            "product_url_raw"
        ],

        "right_retailer": right_retailer,
        "right_source_record_id": right_id,
        "right_product_name": right_row[
            "product_name_raw"
        ],
        "right_name_match_text": right_row[
            "name_match_text"
        ],
        "right_specification": right_row[
            "specification_raw"
        ],
        "right_pack_count": right_row[
            "pack_count"
        ],
        "right_total_quantity_base": right_row[
            "total_quantity_base"
        ],
        "right_base_unit": right_row[
            "base_unit"
        ],
        "right_product_price": right_row[
            "product_price"
        ],
        "right_stock_status": right_row[
            "stock_status"
        ],
        "right_offer_eligible": right_row[
            "offer_eligible"
        ],
        "right_product_url": right_row[
            "product_url_raw"
        ],

        "token_jaccard": round(
            float(jaccard),
            6
        ),
        "sequence_similarity": round(
            float(sequence),
            6
        ),
        "combined_similarity": round(
            float(combined),
            6
        ),
        "exact_name_core": bool(
            exact_core
        ),
        "candidate_priority": priority,
    }


candidate_records = []

grouped = pilot2b.groupby(
    STRICT_BLOCK_COLUMNS,
    dropna=False,
    sort=False,
)

for block_key, block in grouped:
    retailers_here = set(
        block["retailer"]
        .unique()
    )

    if len(retailers_here) < 2:
        continue

    for (
        left_retailer,
        right_retailer,
    ) in RETAILER_PAIRS:
        if (
            left_retailer
            not in retailers_here
            or right_retailer
            not in retailers_here
        ):
            continue

        left_df = block.loc[
            block["retailer"]
            == left_retailer
        ]

        right_df = block.loc[
            block["retailer"]
            == right_retailer
        ]

        potential_pairs = (
            len(left_df)
            * len(right_df)
        )

        large_block = (
            potential_pairs
            > MAX_CARTESIAN_PAIRS_PER_BLOCK
        )

        for _, left_row in left_df.iterrows():
            local_candidates = []

            for _, right_row in right_df.iterrows():
                if (
                    large_block
                    and not shared_meaningful_token(
                        left_row[
                            "name_match_text"
                        ],
                        right_row[
                            "name_match_text"
                        ],
                    )
                ):
                    continue

                record = make_pair_record(
                    left_row,
                    right_row,
                    left_retailer,
                    right_retailer,
                )

                if (
                    record[
                        "combined_similarity"
                    ]
                    >= MIN_COMBINED_SIMILARITY_TO_KEEP
                    or record[
                        "exact_name_core"
                    ]
                ):
                    local_candidates.append(
                        record
                    )

            # Keep the best candidates for this left listing.
            local_candidates = sorted(
                local_candidates,
                key=lambda row: (
                    row[
                        "exact_name_core"
                    ],
                    row[
                        "combined_similarity"
                    ],
                ),
                reverse=True,
            )[
                :MAX_CANDIDATES_PER_SOURCE_PER_PAIR
            ]

            candidate_records.extend(
                local_candidates
            )


strict_candidates = pd.DataFrame(
    candidate_records,
    columns=PAIR_OUTPUT_COLUMNS,
)

# A right listing may appear for many left listings.
# Keep all unique listing pairs but remove accidental duplicates.
if not strict_candidates.empty:
    strict_candidates = (
        strict_candidates
        .drop_duplicates(
            subset=[
                "candidate_pair_id"
            ]
        )
        .reset_index(drop=True)
    )

print(
    "Strict candidate pairs generated:",
    f"{len(strict_candidates):,}"
)

display(
    strict_candidates
    .sort_values(
        [
            "candidate_priority",
            "combined_similarity",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .head(20)
)


Strict candidate pairs generated: 10,650


,candidate_pair_id,retailer_pair,target_category,brand_normalized,strict_pack_key,left_retailer,left_source_record_id,left_product_name,left_name_match_text,left_specification,...,right_base_unit,right_product_price,right_stock_status,right_offer_eligible,right_product_url,token_jaccard,sequence_similarity,combined_similarity,exact_name_core,candidate_priority
47,redmart_00000894__fairprice_00038596,redmart__fairprice,Snacks,fairy port,g|1|400.000000|400.000000,redmart,redmart_00000894,Fairy Port Frozen Musang King Durian Mille Cre...,frozen musang king durian mille crepe cake 6 i...,400 g,...,g,28.99,IN_STOCK,True,https://www.fairprice.com.sg/product/fairy-por...,1.000000,0.933333,0.973333,False,HIGH
2096,redmart_00010264__fairprice_00060297,redmart__fairprice,Drinks,kirin,ml|1|500.000000|500.000000,redmart,redmart_00010264,Kirin Afternoon Milk Tea 500ML,afternoon milk tea,500 ml,...,ml,2.90,OUT_OF_STOCK,False,https://www.fairprice.com.sg/product/kirin-aft...,1.000000,0.900000,0.960000,False,HIGH
6141,redmart_00016081__fairprice_00001437,redmart__fairprice,Snacks,munchy's,g|1|189.000000|189.000000,redmart,redmart_00016081,Munchy's Lexus Cookies Dark Chocolate Cookies ...,lexus cookies dark chocolate cookies,189 g,...,g,3.63,IN_STOCK,True,https://www.fairprice.com.sg/product/munchy-s-...,1.000000,0.875000,0.950000,False,HIGH
6150,redmart_00016081__coldstorage_00007661,redmart__coldstorage,Snacks,munchy's,g|1|189.000000|189.000000,redmart,redmart_00016081,Munchy's Lexus Cookies Dark Chocolate Cookies ...,lexus cookies dark chocolate cookies,189 g,...,g,3.65,IN_STOCK,True,https://coldstorage.com.sg/product/lexus-cooki...,1.000000,0.875000,0.950000,False,HIGH
2361,fairprice_00059429__coldstorage_00004200,fairprice__coldstorage,Drinks,ufc,ml|1|1000.000000|1000.000000,fairprice,fairprice_00059429,UFC Refresh Coconut Twist Water,refresh coconut twist water,1L,...,ml,4.30,OUT_OF_STOCK,False,https://coldstorage.com.sg/product/ufc-refresh...,1.000000,0.870968,0.948387,False,HIGH
8906,redmart_00035261__fairprice_00066534,redmart__fairprice,Canned & Packaged Food,lee kum kee,g|1|480.000000|480.000000,redmart,redmart_00035261,Lee Kum Kee Squeezy Bottle Panda Oyster Sauce ...,squeezy bottle panda oyster sauce reduced salt,480 g,...,g,5.55,IN_STOCK,True,https://www.fairprice.com.sg/product/lee-kum-k...,1.000000,0.869565,0.947826,False,HIGH
6144,redmart_00016082__fairprice_00001409,redmart__fairprice,Snacks,munchy's,g|1|189.000000|189.000000,redmart,redmart_00016082,Munchy's Lexus Cookies Mixed Nuts Cookies 189G,lexus cookies mixed nuts cookies,189 g,...,g,3.63,IN_STOCK,True,https://www.fairprice.com.sg/product/munchy-s-...,1.000000,0.857143,0.942857,False,HIGH
6153,redmart_00016082__coldstorage_00007662,redmart__coldstorage,Snacks,munchy's,g|1|189.000000|189.000000,redmart,redmart_00016082,Munchy's Lexus Cookies Mixed Nuts Cookies 189G,lexus cookies mixed nuts cookies,189 g,...,g,3.65,IN_STOCK,True,https://coldstorage.com.sg/product/munchys-lex...,1.000000,0.857143,0.942857,False,HIGH
293,redmart_00003079__fairprice_00060354,redmart__fairprice,Drinks,giovannini,g|1|1000.000000|1000.000000,redmart,redmart_00003079,Giovannini Speciale N.1 - Fresh Espresso Coffe...,speciale n 1 fresh espresso coffee beans,1 kg,...,g,41.90,IN_STOCK,True,https://www.fairprice.com.sg/product/giovannin...,1.000000,0.850000,0.940000,False,HIGH
6514,redmart_00018516__fairprice_00012489,redmart__fairprice,Snacks,lindt,g|1|100.000000|100.000000,redmart,redmart_00018516,Lindt Swiss Classic Milk Chocolate Bar 100G,swiss classic milk chocolate bar,100 g,...,g,7.60,IN_STOCK,True,https://www.fairprice.com.sg/product/lindt-swi...,1.000000,0.843750,0.937500,False,HIGH


## 14. Assign explicit review recommendation

In [14]:
if len(strict_candidates) > 0:
    strict_candidates[
        "review_recommendation"
    ] = pd.Series(
        np.select(
            [
                strict_candidates[
                    "exact_name_core"
                ].eq(True),

                (
                    strict_candidates[
                        "token_jaccard"
                    ] >= 0.80
                )
                &
                (
                    strict_candidates[
                        "sequence_similarity"
                    ] >= 0.80
                ),

                strict_candidates[
                    "combined_similarity"
                ] >= 0.65,
            ],
            [
                "EXACT_NAME_REVIEW",
                "STRONG_VARIANT_REVIEW",
                "POSSIBLE_MATCH_REVIEW",
            ],
            default="LOW_SIMILARITY_REVIEW",
        ),
        index=strict_candidates.index,
        dtype="string",
    )

    strict_candidates[
        "final_match_verified"
    ] = pd.Series(
        pd.NA,
        index=strict_candidates.index,
        dtype="boolean",
    )

    strict_candidates[
        "verification_notes"
    ] = pd.Series(
        pd.NA,
        index=strict_candidates.index,
        dtype="string",
    )

print(
    "Review recommendation counts:"
)

if len(strict_candidates) > 0:
    print(
        strict_candidates[
            "review_recommendation"
        ]
        .value_counts()
    )


Review recommendation counts:
review_recommendation
LOW_SIMILARITY_REVIEW    8424
POSSIBLE_MATCH_REVIEW    1368
EXACT_NAME_REVIEW         647
STRONG_VARIANT_REVIEW     211
Name: count, dtype: Int64


## 15. Build high-priority review queue

These are the pairs most worth reviewing first.

They are **not automatically verified matches**.


In [15]:
if len(strict_candidates) > 0:
    high_priority_review = (
        strict_candidates.loc[
            strict_candidates[
                "review_recommendation"
            ].isin(
                [
                    "EXACT_NAME_REVIEW",
                    "STRONG_VARIANT_REVIEW",
                ]
            )
        ]
        .copy()
        .sort_values(
            [
                "target_category",
                "brand_normalized",
                "combined_similarity",
            ],
            ascending=[
                True,
                True,
                False,
            ]
        )
    )
else:
    high_priority_review = (
        strict_candidates.copy()
    )

print(
    "High-priority review pairs:",
    f"{len(high_priority_review):,}"
)

display(
    high_priority_review.head(30)
)


High-priority review pairs: 858


,candidate_pair_id,retailer_pair,target_category,brand_normalized,strict_pack_key,left_retailer,left_source_record_id,left_product_name,left_name_match_text,left_specification,...,right_offer_eligible,right_product_url,token_jaccard,sequence_similarity,combined_similarity,exact_name_core,candidate_priority,review_recommendation,final_match_verified,verification_notes
8053,redmart_00034427__fairprice_00066300,redmart__fairprice,Canned & Packaged Food,aaa,g|1|150.000000|150.000000,redmart,redmart_00034427,AAA Sriracha Chilli Sauce,sriracha chilli sauce,150 g,...,True,https://www.fairprice.com.sg/product/aaa-srira...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>
8098,redmart_00033707__fairprice_00066401,redmart__fairprice,Canned & Packaged Food,aaa,ml|1|400.000000|400.000000,redmart,redmart_00033707,AAA Sugar Free Black Soybean Soy Sauce,sugar free black soybean soy sauce,400 ml,...,True,https://www.fairprice.com.sg/product/aaa-sugar...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>
8103,redmart_00033733__fairprice_00066465,redmart__fairprice,Canned & Packaged Food,aaa,ml|1|400.000000|400.000000,redmart,redmart_00033733,AAA Japanese Style Brewed Soy Sauce,japanese style brewed soy sauce,400 ml,...,True,https://www.fairprice.com.sg/product/aaa-japan...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>
8123,fairprice_00010371__coldstorage_00002259,fairprice__coldstorage,Canned & Packaged Food,aaa,g|1|170.000000|170.000000,fairprice,fairprice_00010371,AAA Pickled Mustard,pickled mustard,170 G,...,True,https://coldstorage.com.sg/product/aaa-pickled...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>
8683,redmart_00034411__fairprice_00066301,redmart__fairprice,Canned & Packaged Food,aaa,g|1|793.000000|793.000000,redmart,redmart_00034411,AAA Sriracha Chilli Sauce,sriracha chilli sauce,793 g,...,True,https://www.fairprice.com.sg/product/aaa-srira...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>
8893,redmart_00035259__fairprice_00066547,redmart__fairprice,Canned & Packaged Food,aaa,g|1|510.000000|510.000000,redmart,redmart_00035259,AAA Premium Oyster Sauce With Dried Scallop,premium oyster sauce dried scallop,510 g,...,True,https://www.fairprice.com.sg/product/aaa-premi...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>
8896,redmart_00035264__fairprice_00066543,redmart__fairprice,Canned & Packaged Food,aaa,g|1|510.000000|510.000000,redmart,redmart_00035264,AAA Premium Oyster Sauce,premium oyster sauce,510 g,...,True,https://www.fairprice.com.sg/product/aaa-premi...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>
8899,redmart_00035269__fairprice_00066549,redmart__fairprice,Canned & Packaged Food,aaa,g|1|510.000000|510.000000,redmart,redmart_00035269,AAA Vegetarian Oyster Sauce,vegetarian oyster sauce,510 g,...,True,https://www.fairprice.com.sg/product/aaa-veget...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>
8119,redmart_00033754__fairprice_00066464,redmart__fairprice,Canned & Packaged Food,aaa,ml|1|300.000000|300.000000,redmart,redmart_00033754,AAA 40 - percent Less Sodium Lite Soy Sauce,40 percent less sodium lite soy sauce,300 ml,...,True,https://www.fairprice.com.sg/product/aaa-40-le...,0.857143,0.878788,0.865801,False,HIGH,STRONG_VARIANT_REVIEW,<NA>,<NA>
8549,redmart_00033704__fairprice_00066405,redmart__fairprice,Canned & Packaged Food,amocan,ml|1|640.000000|640.000000,redmart,redmart_00033704,Amocan Golden Label Dark Soya Sauce,golden label dark soya sauce,640 ml,...,True,https://www.fairprice.com.sg/product/amocan-go...,1.000000,1.000000,1.000000,True,VERY_HIGH,EXACT_NAME_REVIEW,<NA>,<NA>


## 16. Generate relaxed total-quantity candidates for pack discrepancies

This path catches cases where two listings have:

- the same target category;
- the same normalized brand;
- the same total base quantity;

but **different strict pack structures**.

These are not exact pack matches and therefore cannot be accepted without manual inspection.


In [16]:
RELAXED_BLOCK_COLUMNS = [
    "target_category_final",
    "brand_normalized",
    "total_pack_key",
]

pack_discrepancy_records = []

relaxed_grouped = pilot2b.groupby(
    RELAXED_BLOCK_COLUMNS,
    dropna=False,
    sort=False,
)

MAX_RELAXED_PAIRS_PER_BLOCK = 2_000
MAX_RELAXED_CANDIDATES_PER_SOURCE = 3

for block_key, block in relaxed_grouped:
    retailers_here = set(
        block["retailer"]
        .unique()
    )

    if len(retailers_here) < 2:
        continue

    # Only blocks containing more than one strict pack
    # representation are relevant here.
    if (
        block[
            "strict_pack_key"
        ].nunique()
        <= 1
    ):
        continue

    for (
        left_retailer,
        right_retailer,
    ) in RETAILER_PAIRS:
        if (
            left_retailer
            not in retailers_here
            or right_retailer
            not in retailers_here
        ):
            continue

        left_df = block.loc[
            block["retailer"]
            == left_retailer
        ]

        right_df = block.loc[
            block["retailer"]
            == right_retailer
        ]

        large_block = (
            len(left_df)
            * len(right_df)
            > MAX_RELAXED_PAIRS_PER_BLOCK
        )

        for _, left_row in left_df.iterrows():
            local_candidates = []

            for _, right_row in right_df.iterrows():
                # We want only strict-pack discrepancies.
                if (
                    left_row[
                        "strict_pack_key"
                    ]
                    == right_row[
                        "strict_pack_key"
                    ]
                ):
                    continue

                if (
                    large_block
                    and not shared_meaningful_token(
                        left_row[
                            "name_match_text"
                        ],
                        right_row[
                            "name_match_text"
                        ],
                    )
                ):
                    continue

                (
                    jaccard,
                    sequence,
                    combined,
                    exact_core,
                ) = score_name_pair(
                    left_row[
                        "name_match_text"
                    ],
                    right_row[
                        "name_match_text"
                    ],
                )

                if combined < 0.50:
                    continue

                local_candidates.append(
                    {
                        "retailer_pair": (
                            f"{left_retailer}"
                            f"__{right_retailer}"
                        ),
                        "target_category": left_row[
                            "target_category_final"
                        ],
                        "brand_normalized": left_row[
                            "brand_normalized"
                        ],
                        "left_source_record_id": left_row[
                            "source_record_id"
                        ],
                        "left_product_name": left_row[
                            "product_name_raw"
                        ],
                        "left_specification": left_row[
                            "specification_raw"
                        ],
                        "left_strict_pack_key": left_row[
                            "strict_pack_key"
                        ],
                        "right_source_record_id": right_row[
                            "source_record_id"
                        ],
                        "right_product_name": right_row[
                            "product_name_raw"
                        ],
                        "right_specification": right_row[
                            "specification_raw"
                        ],
                        "right_strict_pack_key": right_row[
                            "strict_pack_key"
                        ],
                        "total_pack_key": left_row[
                            "total_pack_key"
                        ],
                        "token_jaccard": round(
                            float(jaccard),
                            6
                        ),
                        "sequence_similarity": round(
                            float(sequence),
                            6
                        ),
                        "combined_similarity": round(
                            float(combined),
                            6
                        ),
                        "review_reason": (
                            "SAME_BRAND_AND_TOTAL_QUANTITY_"
                            "BUT_PACK_STRUCTURE_DIFFERS"
                        ),
                    }
                )

            local_candidates = sorted(
                local_candidates,
                key=lambda row: row[
                    "combined_similarity"
                ],
                reverse=True,
            )[
                :MAX_RELAXED_CANDIDATES_PER_SOURCE
            ]

            pack_discrepancy_records.extend(
                local_candidates
            )


pack_discrepancy_review = pd.DataFrame(
    pack_discrepancy_records
)

if not pack_discrepancy_review.empty:
    pack_discrepancy_review = (
        pack_discrepancy_review
        .drop_duplicates(
            subset=[
                "left_source_record_id",
                "right_source_record_id",
            ]
        )
        .reset_index(drop=True)
    )

print(
    "Pack-discrepancy review candidates:",
    f"{len(pack_discrepancy_review):,}"
)

display(
    pack_discrepancy_review.head(30)
)


Pack-discrepancy review candidates: 281


,retailer_pair,target_category,brand_normalized,left_source_record_id,left_product_name,left_specification,left_strict_pack_key,right_source_record_id,right_product_name,right_specification,right_strict_pack_key,total_pack_key,token_jaccard,sequence_similarity,combined_similarity,review_reason
0,redmart__fairprice,Drinks,english tea shop,redmart_00001300,English Tea Shop Organic Lemongrass Ginger And...,30 g,g|1|30.000000|30.000000,fairprice_00000048,English Tea Shop Organic Tea - Lemongrass Ging...,20 x 1.5g,g|20|1.500000|30.000000,g|30.000000,0.833333,0.842105,0.836842,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
1,redmart__fairprice,Drinks,twinings,redmart_00001992,Twinings Pure Green Tea 25's,50 g,g|1|50.000000|50.000000,fairprice_00059835,Twinings Teabags - Pure Green Tea,25 x 2g,g|25|2.000000|50.000000,g|50.000000,0.500000,0.682927,0.573171,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
2,redmart__coldstorage,Drinks,twinings,redmart_00001998,Twinings English Breakfast Tea 25's,50 g,g|1|50.000000|50.000000,coldstorage_00001161,Twinings English Breakfast Teabags 25 x 2g,25 x 2g,g|25|2.000000|50.000000,g|50.000000,0.333333,0.862745,0.545098,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
3,redmart__coldstorage,Drinks,gingen,redmart_00001399,Gingen Ginger with Brown Sugar Natural Beverage,180 g,g|1|180.000000|180.000000,coldstorage_00001059,Gingen Ginger with Brown Sugar Tea Powder 10 x...,10 x 18g,g|10|18.000000|180.000000,g|180.000000,0.428571,0.718750,0.544643,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
4,redmart__coldstorage,Drinks,gingen,redmart_00001508,Gingen Ginger Tea Strong Formula,180 g,g|1|180.000000|180.000000,coldstorage_00001062,Gingen Strong Ginger Tea Powder 10 x 18g,10 x 18g,g|10|18.000000|180.000000,g|180.000000,0.600000,0.489796,0.555918,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
5,redmart__coldstorage,Drinks,teekanne,redmart_00001511,Teekanne Ginger Lemon Tea 20's,35 g,g|1|35.000000|35.000000,coldstorage_00001152,Teekanne Ginger Lemon Tea Bags 20 x 1.75g,20 x 1.75g,g|20|1.750000|35.000000,g|35.000000,0.500000,0.857143,0.642857,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
6,redmart__coldstorage,Drinks,teekanne,redmart_00001418,Teekanne Peppermint Tea 20's,45 g,g|1|45.000000|45.000000,coldstorage_00001154,Teekanne Peppermint Tea Bags 20 x 2.25g,20 x 2.25g,g|20|2.250000|45.000000,g|45.000000,0.400000,0.842105,0.576842,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
7,redmart__fairprice,Drinks,lipton,redmart_00001949,Lipton Yellow Label Tea,100 × 2 g,g|100|2.000000|200.000000,fairprice_00060284,Lipton Yellow Label Tea Bags - International B...,200 G,g|1|200.000000|200.000000,g|200.000000,0.500000,0.561404,0.524561,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
8,redmart__fairprice,Drinks,english tea shop,redmart_00001950,English Tea Shop Organic Rooibos Tea 20s,40 g,g|1|40.000000|40.000000,fairprice_00000051,English Tea Shop Organic Tea - Rooibos,20 x 2g,g|20|2.000000|40.000000,g|40.000000,0.750000,0.714286,0.735714,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...
9,redmart__coldstorage,Drinks,twinings,redmart_00001951,Twinings English Breakfast Tea (50's),100 g,g|1|100.000000|100.000000,coldstorage_00001162,Twinings English Breakfast Teabags 50 x 2g,50 x 2g,g|50|2.000000|100.000000,g|100.000000,0.333333,0.862745,0.545098,SAME_BRAND_AND_TOTAL_QUANTITY_BUT_PACK_STRUCTU...


## 17. Identify pilot records with at least one strict candidate

In [17]:
candidate_record_ids = set()

if len(strict_candidates) > 0:
    candidate_record_ids.update(
        strict_candidates[
            "left_source_record_id"
        ].astype(str)
    )

    candidate_record_ids.update(
        strict_candidates[
            "right_source_record_id"
        ].astype(str)
    )

pilot2b[
    "has_strict_cross_retailer_candidate"
] = (
    pilot2b[
        "source_record_id"
    ]
    .astype(str)
    .isin(candidate_record_ids)
)

print(
    "Pilot records with >=1 strict candidate:",
    f"{int(pilot2b['has_strict_cross_retailer_candidate'].sum()):,}"
)

print(
    "Pilot records with no strict candidate:",
    f"{int((~pilot2b['has_strict_cross_retailer_candidate']).sum()):,}"
)


Pilot records with >=1 strict candidate: 6,615
Pilot records with no strict candidate: 24,980


## 18. Create unmatched pilot queue

In [18]:
unmatched_pilot = (
    pilot2b.loc[
        ~pilot2b[
            "has_strict_cross_retailer_candidate"
        ]
    ]
    .copy()
)

print(
    "Unmatched pilot records:",
    f"{len(unmatched_pilot):,}"
)

display(
    unmatched_pilot[
        [
            "target_category_final",
            "retailer",
            "source_record_id",
            "product_name_raw",
            "brand_normalized",
            "specification_raw",
            "strict_pack_key",
            "stock_status",
            "offer_eligible",
            "product_url_raw",
        ]
    ].head(30)
)


Unmatched pilot records: 24,980


,target_category_final,retailer,source_record_id,product_name_raw,brand_normalized,specification_raw,strict_pack_key,stock_status,offer_eligible,product_url_raw
0,Snacks,redmart,redmart_00000002,Sojasun Natural Soya Yoghurt,sojasun,6 × 100 g,g|6|100.000000|600.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i303336401-...
1,Snacks,redmart,redmart_00000003,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix,4 × 125 g,g|4|125.000000|500.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i462726209-...
2,Snacks,redmart,redmart_00000004,Top Gourmet Cheng Tng,top gourmet,400 g,g|1|400.000000|400.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i303250731-...
3,Snacks,redmart,redmart_00000005,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,la laitiere,4 × 115 g,g|4|115.000000|460.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i301170176-...
4,Snacks,redmart,redmart_00000006,Monoprix Flan Caramel Vanilla Pudding 4X100G,monoprix,4 × 100 g,g|4|100.000000|400.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i1711630409...
5,Snacks,redmart,redmart_00000007,Fortune Sweetened Chin Chow 1KG,fortune,1 kg,g|1|1000.000000|1000.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i301172144-...
6,Snacks,redmart,redmart_00000008,RedMart Trio Swiss Roll (Singapore Special),redmart,430 g,g|1|430.000000|430.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i1931806406...
10,Snacks,redmart,redmart_00000012,Sainsbury's Tiramisu (Fresh From Italy),sainsbury's,500 g,g|1|500.000000|500.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i3451293658...
11,Snacks,redmart,redmart_00000013,Fortune Pulut Hitam Black Glutinous Rice 380G,fortune,380 g,g|1|380.000000|380.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i2740814436...
12,Snacks,redmart,redmart_00000014,Top Gourmet Yam Paste With Ginkgo Nuts,top gourmet,250 g,g|1|250.000000|250.000000,IN_STOCK,True,https://www.lazada.sg/products/pdp-i303250733-...


## 19. Create candidate summary by category and retailer pair

This helps assess whether the pilot has enough cross-retailer overlap before manual verification.


In [19]:
if len(strict_candidates) > 0:
    candidate_summary = (
        strict_candidates
        .groupby(
            [
                "target_category",
                "retailer_pair",
                "review_recommendation",
            ],
            dropna=False,
        )
        .size()
        .reset_index(
            name="candidate_pair_count"
        )
        .sort_values(
            [
                "target_category",
                "retailer_pair",
                "candidate_pair_count",
            ],
            ascending=[
                True,
                True,
                False,
            ]
        )
    )
else:
    candidate_summary = pd.DataFrame(
        columns=[
            "target_category",
            "retailer_pair",
            "review_recommendation",
            "candidate_pair_count",
        ]
    )

display(candidate_summary)


,target_category,retailer_pair,review_recommendation,candidate_pair_count
1,Canned & Packaged Food,fairprice__coldstorage,LOW_SIMILARITY_REVIEW,333
2,Canned & Packaged Food,fairprice__coldstorage,POSSIBLE_MATCH_REVIEW,70
0,Canned & Packaged Food,fairprice__coldstorage,EXACT_NAME_REVIEW,22
3,Canned & Packaged Food,fairprice__coldstorage,STRONG_VARIANT_REVIEW,6
5,Canned & Packaged Food,redmart__coldstorage,LOW_SIMILARITY_REVIEW,256
6,Canned & Packaged Food,redmart__coldstorage,POSSIBLE_MATCH_REVIEW,71
4,Canned & Packaged Food,redmart__coldstorage,EXACT_NAME_REVIEW,46
7,Canned & Packaged Food,redmart__coldstorage,STRONG_VARIANT_REVIEW,3
9,Canned & Packaged Food,redmart__fairprice,LOW_SIMILARITY_REVIEW,547
8,Canned & Packaged Food,redmart__fairprice,EXACT_NAME_REVIEW,96


## 20. Create strict-block summary with candidate coverage

In [20]:
candidate_block_keys = set()

if len(strict_candidates) > 0:
    candidate_block_keys = set(
        zip(
            strict_candidates[
                "target_category"
            ],
            strict_candidates[
                "brand_normalized"
            ],
            strict_candidates[
                "strict_pack_key"
            ],
        )
    )

strict_block_summary[
    "has_kept_candidate_pair"
] = strict_block_summary.apply(
    lambda row: (
        row[
            "target_category_final"
        ],
        row[
            "brand_normalized"
        ],
        row[
            "strict_pack_key"
        ],
    ) in candidate_block_keys,
    axis=1,
)

display(
    strict_block_summary
    .sort_values(
        [
            "retailer_count",
            "total_records",
        ],
        ascending=[
            False,
            False,
        ]
    )
    .head(30)
)


,target_category_final,brand_normalized,strict_pack_key,total_records,retailer_count,redmart_records,fairprice_records,coldstorage_records,has_kept_candidate_pair
6012,Drinks,pokka,ml|1|1500.000000|1500.000000,56,3,21,20,15,True
6024,Drinks,pokka,ml|6|250.000000|1500.000000,55,3,22,23,10,True
6016,Drinks,pokka,ml|1|500.000000|500.000000,49,3,14,3,32,True
10347,Snacks,lindt,g|1|100.000000|100.000000,43,3,25,9,9,True
3901,Drinks,boncafe,g|1|200.000000|200.000000,34,3,11,9,14,True
184,Canned & Packaged Food,ayam brand,g|1|150.000000|150.000000,33,3,2,15,16,True
9232,Snacks,haribo,g|1|160.000000|160.000000,33,3,14,7,12,True
7040,Drinks,yeo's,ml|6|250.000000|1500.000000,31,3,3,17,11,True
7834,Snacks,cadbury,g|1|180.000000|180.000000,31,3,11,8,12,True
10212,Snacks,lay's,g|1|170.000000|170.000000,30,3,10,9,11,True


## 21. Validate candidate-pair invariants

In [21]:
if len(strict_candidates) > 0:
    assert (
        strict_candidates[
            "left_source_record_id"
        ]
        != strict_candidates[
            "right_source_record_id"
        ]
    ).all()

    assert (
        strict_candidates[
            "left_retailer"
        ]
        != strict_candidates[
            "right_retailer"
        ]
    ).all()

    assert (
        strict_candidates[
            "candidate_pair_id"
        ]
        .duplicated()
        .sum()
        == 0
    )

    assert (
        strict_candidates[
            "brand_normalized"
        ]
        .notna()
        .all()
    )

    assert (
        strict_candidates[
            "strict_pack_key"
        ]
        .notna()
        .all()
    )

    assert (
        strict_candidates[
            "combined_similarity"
        ]
        .between(
            0,
            1
        )
        .all()
    )

print(
    "PASS: Strict candidate-pair invariants succeeded."
)


PASS: Strict candidate-pair invariants succeeded.


## 22. Confirm stock does not affect identity matching

Out-of-stock catalog records are allowed to participate in candidate generation.

Their current availability remains visible in the candidate output.


In [22]:
if len(strict_candidates) > 0:
    candidate_out_of_stock = (
        (
            strict_candidates[
                "left_stock_status"
            ]
            == "OUT_OF_STOCK"
        )
        |
        (
            strict_candidates[
                "right_stock_status"
            ]
            == "OUT_OF_STOCK"
        )
    )

    candidate_out_of_stock_count = int(
        candidate_out_of_stock.sum()
    )

else:
    candidate_out_of_stock_count = 0


print(
    "Candidate pairs containing at least one "
    "out-of-stock listing:",
    f"{candidate_out_of_stock_count:,}"
)

print(
    "PASS: Stock remains separate from product identity."
)


Candidate pairs containing at least one out-of-stock listing: 1,733
PASS: Stock remains separate from product identity.


## 23. Verify Stage 2A pilot fields remain unchanged

In [23]:
original_columns = list(
    pilot_snapshot.columns
)

assert pilot2b[
    original_columns
].equals(
    pilot_snapshot
), (
    "At least one original Stage 2A pilot field changed."
)

print(
    "PASS: All original Stage 2A pilot fields "
    "remain unchanged."
)


PASS: All original Stage 2A pilot fields remain unchanged.


## 24. Save enriched pilot table

In [24]:
ENRICHED_PILOT_FILE = (
    CURATED_DATA_DIR
    / "stage2b_pilot_enriched.csv"
)

pilot2b.to_csv(
    ENRICHED_PILOT_FILE,
    index=False,
    encoding="utf-8"
)

print(ENRICHED_PILOT_FILE)


D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2b_pilot_enriched.csv


## 25. Save Stage 2B candidate and review files

In [25]:
STRICT_CANDIDATES_FILE = (
    CURATED_DATA_DIR
    / "stage2b_strict_candidate_pairs.csv"
)

HIGH_PRIORITY_FILE = (
    CURATED_DATA_DIR
    / "stage2b_high_priority_review.csv"
)

PACK_DISCREPANCY_FILE = (
    CURATED_DATA_DIR
    / "stage2b_pack_discrepancy_review.csv"
)

UNMATCHED_FILE = (
    CURATED_DATA_DIR
    / "stage2b_unmatched_pilot.csv"
)

CANDIDATE_SUMMARY_FILE = (
    CURATED_DATA_DIR
    / "stage2b_candidate_summary.csv"
)

BLOCK_SUMMARY_FILE = (
    CURATED_DATA_DIR
    / "stage2b_strict_block_summary.csv"
)

strict_candidates.to_csv(
    STRICT_CANDIDATES_FILE,
    index=False,
    encoding="utf-8"
)

high_priority_review.to_csv(
    HIGH_PRIORITY_FILE,
    index=False,
    encoding="utf-8"
)

pack_discrepancy_review.to_csv(
    PACK_DISCREPANCY_FILE,
    index=False,
    encoding="utf-8"
)

unmatched_pilot.to_csv(
    UNMATCHED_FILE,
    index=False,
    encoding="utf-8"
)

candidate_summary.to_csv(
    CANDIDATE_SUMMARY_FILE,
    index=False,
    encoding="utf-8"
)

strict_block_summary.to_csv(
    BLOCK_SUMMARY_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved Stage 2B outputs:")
for file_path in [
    ENRICHED_PILOT_FILE,
    STRICT_CANDIDATES_FILE,
    HIGH_PRIORITY_FILE,
    PACK_DISCREPANCY_FILE,
    UNMATCHED_FILE,
    CANDIDATE_SUMMARY_FILE,
    BLOCK_SUMMARY_FILE,
]:
    print("-", file_path)


Saved Stage 2B outputs:
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2b_pilot_enriched.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2b_strict_candidate_pairs.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2b_high_priority_review.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2b_pack_discrepancy_review.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2b_unmatched_pilot.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2b_candidate_summary.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2b_strict_block_summary.csv


## 26. Verify Stage 2B output files

In [26]:
OUTPUT_FILES = [
    ENRICHED_PILOT_FILE,
    STRICT_CANDIDATES_FILE,
    HIGH_PRIORITY_FILE,
    PACK_DISCREPANCY_FILE,
    UNMATCHED_FILE,
    CANDIDATE_SUMMARY_FILE,
    BLOCK_SUMMARY_FILE,
]

for file_path in OUTPUT_FILES:
    print(
        f"{file_path.name:48s} -> "
        f"{file_path.exists()}"
    )

    assert file_path.exists()

print(
    "\nPASS: All Stage 2B output files were created."
)


stage2b_pilot_enriched.csv                       -> True
stage2b_strict_candidate_pairs.csv               -> True
stage2b_high_priority_review.csv                 -> True
stage2b_pack_discrepancy_review.csv              -> True
stage2b_unmatched_pilot.csv                      -> True
stage2b_candidate_summary.csv                    -> True
stage2b_strict_block_summary.csv                 -> True

PASS: All Stage 2B output files were created.


## 27. Final Stage 2B validation

In [27]:
strict_pair_count = len(
    strict_candidates
)

high_priority_count = len(
    high_priority_review
)

pack_discrepancy_count = len(
    pack_discrepancy_review
)

matched_record_count = int(
    pilot2b[
        "has_strict_cross_retailer_candidate"
    ].sum()
)

unmatched_record_count = len(
    unmatched_pilot
)

print("=" * 86)
print("STAGE 2B FINAL VALIDATION")
print("=" * 86)

print(
    f"Pilot catalog records                  : "
    f"{len(pilot2b):,}"
)

print(
    f"Pilot categories                       : "
    f"{pilot2b['target_category_final'].nunique():,}"
)

print(
    f"Strict multi-retailer blocks           : "
    f"{len(multi_retailer_blocks):,}"
)

print("-" * 86)

print(
    f"Strict candidate pairs                 : "
    f"{strict_pair_count:,}"
)

print(
    f"High-priority review pairs             : "
    f"{high_priority_count:,}"
)

print(
    f"Pack-discrepancy review pairs          : "
    f"{pack_discrepancy_count:,}"
)

print("-" * 86)

print(
    f"Pilot records with strict candidates   : "
    f"{matched_record_count:,}"
)

print(
    f"Pilot records without strict candidates: "
    f"{unmatched_record_count:,}"
)

print(
    f"Pairs containing out-of-stock listing  : "
    f"{candidate_out_of_stock_count:,}"
)

print("=" * 86)

assert (
    matched_record_count
    + unmatched_record_count
    == len(pilot2b)
)

assert (
    high_priority_count
    <= strict_pair_count
)

assert pilot2b[
    original_columns
].equals(
    pilot_snapshot
)

print()
print(
    "PASS: Stage 2B cross-retailer candidate "
    "generation succeeded."
)

print()
print(
    "Important: candidate pairs are not final verified "
    "canonical matches. Stage 2C should review brand, "
    "variant, exact pack evidence, fulfilment and source "
    "URLs before approving a canonical identity."
)


STAGE 2B FINAL VALIDATION
Pilot catalog records                  : 31,595
Pilot categories                       : 3
Strict multi-retailer blocks           : 1,423
--------------------------------------------------------------------------------------
Strict candidate pairs                 : 10,650
High-priority review pairs             : 858
Pack-discrepancy review pairs          : 281
--------------------------------------------------------------------------------------
Pilot records with strict candidates   : 6,615
Pilot records without strict candidates: 24,980
Pairs containing out-of-stock listing  : 1,733

PASS: Stage 2B cross-retailer candidate generation succeeded.

Important: candidate pairs are not final verified canonical matches. Stage 2C should review brand, variant, exact pack evidence, fulfilment and source URLs before approving a canonical identity.


## Stage 2B result

When the final validation passes:

- candidate generation is blocked on exact normalized brand and exact pack structure;
- product-name similarity is used only to prioritize review;
- equal total quantity with different pack structure is separated into a pack-discrepancy queue;
- stock availability does not remove a listing from product-identity matching;
- no candidate is silently converted into a final verified match;
- unmatched pilot records remain traceable.

### Next stage

Proceed to **Stage 2C: manual verification and canonical identity creation for the pilot**.

Stage 2C should record explicit review decisions such as:

- `MATCH_CONFIRMED`
- `VARIANT_MISMATCH`
- `PACK_MISMATCH`
- `BRAND_MISMATCH`
- `FULFILMENT_NOT_VERIFIED`
- `EXCLUDE_OTHER`

Only confirmed candidate groups should be assigned canonical product IDs.
